# m2mTani-MT: inference

Translate between **Adi, Apatani, Galo, Nyishi, Tagin, Hindi and English** with
[m2mTani-MT](https://huggingface.co/YOUR_HF_USERNAME/m2mTani-MT).

The model is **gated**. Request access on the model page, then log in with a token from the approved account
(`hf auth login`, or set the `HF_TOKEN` environment variable). To use a local copy of the checkpoint instead, set
`MODEL_ID` to its folder.

| Language | Code |
|---|---|
| Adi | `adi_Latn` |
| Apatani | `apt_Latn` |
| Galo | `adl_Latn` |
| Nyishi | `njz_Latn` |
| Tagin | `tgj_Latn` |
| Hindi | `hin_Deva` |
| English | `eng_Latn` |

In [ ]:
# %pip install -U torch "transformers>=5,<6" huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID = "YOUR_HF_USERNAME/m2mTani-MT"        # or a local folder, e.g. "m2m-tani-nllb-32k-steps"

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()
print(f"{model.__class__.__name__} on {DEVICE} ({DTYPE})")

In [ ]:
LANGS = {"English": "eng_Latn", "Hindi": "hin_Deva", "Adi": "adi_Latn", "Apatani": "apt_Latn",
         "Galo": "adl_Latn", "Nyishi": "njz_Latn", "Tagin": "tgj_Latn"}


def translate(texts, src_lang, tgt_lang, num_beams=4, max_new_tokens=256):
    """Translate a list of sentences. src_lang / tgt_lang are codes such as "eng_Latn" or "tgj_Latn"."""
    for code in (src_lang, tgt_lang):
        if code not in LANGS.values():
            raise ValueError(f"{code!r} is not one of the model's languages: {sorted(LANGS.values())}")
    tokenizer.src_lang = src_lang                                      # input = [src_lang] tokens </s>
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**batch, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                             num_beams=num_beams, max_new_tokens=max_new_tokens)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

In [ ]:
sentences = ["Where are you going?", "My mother is cooking rice at home."]
for name in ["Adi", "Apatani", "Galo", "Nyishi", "Tagin", "Hindi"]:
    for s, t in zip(sentences, translate(sentences, "eng_Latn", LANGS[name])):
        print(f"English -> {name:8} {s!r} -> {t!r}")

In [ ]:
# Tani -> Tani and Tani -> English
print(translate(["ngo metikslo kénna amiko."], "adi_Latn", "tgj_Latn"))
print(translate(["ngo metikslo kénna amiko."], "adi_Latn", "eng_Latn"))